In [ ]:
!pip install -U ddgs playwright pandas
!playwright install chromium
!pip install -U pandas ipython
!pip install ftfy

In [ ]:
import re
import asyncio
from urllib.parse import urljoin, urlparse
from playwright.async_api import async_playwright
import pandas as pd
pd.set_option("display.notebook_repr_html", False)
from datetime import datetime
from playwright.async_api import Error as PlaywrightError
from ftfy import fix_text

In [ ]:
async def expand_quora_urls_async(seed_urls, target=150, delay=2.0, headless=True,
                                  scroll_rounds=6, scroll_pause=0.8):

    bad_re = re.compile(r"/profile/|/topic/|/answer/|/signin|/login|/terms|/privacy|/about|/careers|/contact",
                        re.IGNORECASE)

    def looks_like_question(u: str) -> bool:
        p = urlparse(u)
        host = (p.hostname or "").lower()

        # allow ONLY main Quora host
        if host not in ("www.quora.com", "quora.com"):
            return False

        if p.query or p.fragment:
            return False

        if not p.path or p.path == "/":
            return False

        if bad_re.search(p.path):
            return False

        return True

    seen = set(seed_urls)
    queue = list(seed_urls)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        page = await browser.new_page()

        while queue and len(seen) < target:
            url = queue.pop(0)

            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=30000)
                await page.wait_for_timeout(800)

                body = (await page.inner_text("body")).lower()
                if "captcha" in body or ("sign up" in body and "log in" in body):
                    print("Skipping gated page:", url)
                    await asyncio.sleep(delay)
                    continue

                for _ in range(scroll_rounds):
                    await page.mouse.wheel(0, 2500)
                    await page.wait_for_timeout(int(scroll_pause * 1000))

                hrefs = await page.eval_on_selector_all(
                    "a[href]",
                    "els => els.map(e => e.getAttribute('href'))"
                )

                added = 0
                for h in hrefs:
                    if not h:
                        continue

                    abs_u = urljoin("https://www.quora.com", h)
                    if abs_u.endswith("/"):
                        abs_u = abs_u[:-1]

                    if looks_like_question(abs_u) and abs_u not in seen:
                        seen.add(abs_u)
                        queue.append(abs_u)
                        added += 1
                        if len(seen) >= target:
                            break

                print(f"Visited: {url[:60]}... | +{added} | total={len(seen)}")

            except Exception as e:
                print("Error visiting:", url, "|", str(e)[:140])

            await asyncio.sleep(delay)

        await browser.close()

    return list(seen)

In [ ]:
seed = [
    "https://www.quora.com/What-is-your-opinion-on-using-artificial-intelligence-AI-for-teaching-and-learning-Do-you-believe-it-will-enhance-or-diminish-the-quality-of-education",
    "https://www.quora.com/How-can-college-teachers-stop-students-from-using-AI-for-homework",
    "https://www.quora.com/What-are-the-potential-risks-of-relying-heavily-on-AI-in-education-and-how-can-they-be-mitigated",
    "https://www.quora.com/Should-students-be-allowed-to-use-ChatGPT-and-other-AI-tools-for-school-projects",
    "https://www.quora.com/Whats-your-opinion-on-AI-in-education",
    "https://www.quora.com/What-are-the-pros-and-cons-of-using-AI-in-education",
    "https://www.quora.com/What-are-the-benefits-of-artificial-intelligence-in-education-How-can-AI-help-students-learn-better-than-humans",
    "https://www.quora.com/How-can-AI-be-used-to-enhance-education-and-learning",
    "https://www.quora.com/How-are-teachers-actually-using-AI-in-their-classrooms-right-now",
    "https://www.quora.com/Is-AI-useful-for-students-and-education"
]

expanded_urls = await expand_quora_urls_async(seed, target=150, delay=2.5, headless=False)
len(expanded_urls), expanded_urls[:10]

In [ ]:
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def looks_gated(body_lower: str) -> bool:
    return ("captcha" in body_lower) or ("unusual traffic" in body_lower) or (
        "sign up" in body_lower and "log in" in body_lower
    )

BAD_SUBSTRINGS = [
    "Sponsored by", "Promoted by", "Advertisement", "Related questions",
    "More answers below", "Skip to content", "Skip to search",
    "Sign In", "Sign Up", "Log in", "© Quora", "Your response is private",
    "Was this worth your time?"
]
AUTHOR_BIO_RE = re.compile(r"(Author has \d|answer views|Originally Answered)", re.IGNORECASE)

def is_noise(t: str) -> bool:
    if any(b.lower() in t.lower() for b in BAD_SUBSTRINGS):
        return True
    if AUTHOR_BIO_RE.search(t):
        return True
    if t.count("Upvote") >= 2:
        return True
    return False

async def scrape_quora_robust(
    urls,
    out_csv="quora_answers_robust.csv",
    target_records=2000,
    max_answers_per_url=50,
    min_chars=250,
    scroll_rounds=5,
    scroll_pause=0.25,
    delay_between_pages=1.0,
    headless=True,
):
    rows, seen = [], set()

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            )
        )

        # block heavy resources
        async def route_handler(route):
            try:
                if route.request.resource_type in ("image", "media", "font"):
                    await route.abort()
                else:
                    await route.continue_()
            except Exception:
                # If the page/context closes mid-request, ignore it.
                return

        await context.route("**/*", route_handler)

        page = await context.new_page()

        async def ensure_page_alive():
            nonlocal page
            try:
                _ = page.url  # touch it
            except Exception:
                page = await context.new_page()

        for i, url in enumerate(urls, start=1):
            if len(rows) >= target_records:
                break

            await ensure_page_alive()

            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=25000)
                await page.wait_for_timeout(500)

                body = (await page.inner_text("body")).lower()
                if looks_gated(body):
                    print(f"[SKIP gated] {i}/{len(urls)}")
                    await asyncio.sleep(delay_between_pages)
                    continue

                # Title (best effort)
                try:
                    title = norm(await page.inner_text("h1"))
                except Exception:
                    title = ""

                # small scroll
                for _ in range(scroll_rounds):
                    await page.mouse.wheel(0, 2200)
                    await page.wait_for_timeout(int(scroll_pause * 1000))

                # Expand "Continue Reading"
                try:
                    await page.evaluate("""
                    () => {
                      const els = Array.from(document.querySelectorAll('a,button,div[role="button"]'));
                      for (const el of els) {
                        const t = (el.innerText || '').trim().toLowerCase();
                        if (t === 'continue reading') el.click();
                      }
                    }
                    """)
                    await page.wait_for_timeout(300)
                except Exception:
                    pass

                # answer containers -> q-text inside each
                answers = await page.evaluate("""
                () => {
                  const articles = Array.from(document.querySelectorAll('div[role="article"]'));
                  const out = [];
                  for (const a of articles) {
                    const parts = Array.from(a.querySelectorAll('div.q-text, span.q-text'))
                      .map(e => (e.innerText || '').replace(/\\s+/g,' ').trim())
                      .filter(t => t.length > 0);
                    if (parts.length) out.push(parts.join(' '));
                  }
                  return out;
                }
                """)

                # Fallback: if Quora layout doesn't use role=article, grab q-text blocks (less clean, but better than 0)
                if not answers:
                    answers = await page.evaluate("""
                    () => {
                      const els = Array.from(document.querySelectorAll('div.q-text, span.q-text'));
                      return els.map(e => (e.innerText || '').replace(/\\s+/g,' ').trim());
                    }
                    """)

                added = 0
                for t in answers:
                    t = norm(t)
                    if len(t) < min_chars:
                        continue
                    if is_noise(t):
                        continue

                    key = f"{url}::{t[:220]}"
                    if key in seen:
                        continue
                    seen.add(key)

                    rows.append({
                        "source": "quora",
                        "url": url,
                        "question_title": title,
                        "answer_text": t,
                        "scraped_at": datetime.utcnow().isoformat() + "Z",
                    })
                    added += 1
                    if added >= max_answers_per_url or len(rows) >= target_records:
                        break

                print(f"[{i}/{len(urls)}] +{added} | total={len(rows)}")

            except PlaywrightError as e:
                # If the page crashed/closed, recreate it and continue
                msg = str(e)
                print(f"[WARN] Playwright error on {i}/{len(urls)}: {msg[:120]}")
                try:
                    page = await context.new_page()
                except Exception:
                    pass
            except Exception as e:
                print(f"[WARN] Other error on {i}/{len(urls)}: {str(e)[:120]}")

            await asyncio.sleep(delay_between_pages)

        await context.close()
        await browser.close()

    df = pd.DataFrame(rows)
    
    df["answer_text"] = df["answer_text"].apply(fix_text)

    # Normalize whitespace
    df["answer_text"] = df["answer_text"].str.replace(r"\s+", " ", regex=True).str.strip()

    # Drop exact duplicates
    df = df.drop_duplicates(subset=["answer_text"])

    print(f"Final dataset size after dedupe: {len(df)}")

    df.to_csv(out_csv, index=False, encoding="utf-8")
    return df

In [ ]:
df_test = await scrape_quora_robust(expanded_urls, out_csv="quora_crawl.csv", target_records=2000)
df_test.head()